# Spark Application Logs
A proper application must have some kind of application logging.

## How to use Log4j with pyspark?
Configuring Log4j is a three step process :
- Create a Log4j configuration file.
- Configure Spark JVM to pickup the Log4j configuration file.
- Create a python class to get Spark's Log4j instance and use it in the pyspark program

## Log4j properties file
Log4j works almost the same way as python logs.
### Componenets of Log4j : 
- Logger : It is a set of apis which we are going to use in our application.
- Configurations : This is defined in the Log4j properties file and they are loaded with the loggers at run time.
    - Log4j configurations are defined in the hierarchy and the top most hierarchy is the root category.
        - Example : 
        ```python
            # Root logger configuration
            log4j.rootCategory=INFO, console
        ```
            - Here INFO is the log level
    - For any hierarchy or category we define two things first is the log level and the second thing is the list of appenders.
    - Log4j supports multiple log levels such as INFO,DEBUG,WARN,ERROR
    - Here in the top most level info will be shown because we have set INFO to be the root log level.
    - The root logger is the default logger that all other loggers inherit from.
    - If you don’t define a more specific logger (like log4j.logger.org.apache.spark), the root logger decides what happens to log messages.
    - It controls what level of messages are logged and where they go.
- Appender : Appenders are the output destinations such as console and log file. These appenders are also configured in the Log4j property file.
    - Console appender : 
        - Example : 
        ```python
            # Root logger configuration
            log4j.rootCategory=INFO, console
        ```
            - Here console is the appender
- **NOTE :** Hierarchy and Appender section define the root level Log4j configurations and they will stop all the log messages sent by the spark and other packages except Warning and errors.

#### Automatically detect the configuration file and set up the configuration for Log4j for a spark program written in python
In this pyspark application with dynamic project level log4j logging configured here in this example is portable in nature because the log4j.properties file along with the logs folder stays inside the project's folder which makes it extremely portable

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand

# For keeping track of time taken to complete the spark computation task
import time
import sys

# spark Log4j logging related imports
from pyspark import SparkConf
import os

# Logging related Spark configurations setup
# log4k.properties configuration file path setup
# Determine project directory — works in both script & notebook
try:
    project_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in interactive mode (e.g., Jupyter)
    project_dir = os.getcwd()

# Dynamically define your log directory (for example: logs inside project)
log_dir = os.path.join(project_dir, "log4j_properties", "logs")
os.makedirs(log_dir, exist_ok=True)  # ensure it exists

log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")

# Create SparkConf with custom Log4j config
conf = (
    SparkConf()
    .setAppName("CPU_Stress_Test")
    .setMaster("local[*]")
    # JVM property for log4j 
    # Use this Is you have set the directory where the log file must be generated inside the log4j.properties file
    # .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    # .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
    # Use this If you want to setup the directory where the Log files must be generated using python
    .set("spark.driver.extraJavaOptions",
         f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    .set("spark.executor.extraJavaOptions",
         f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    # Ensure executors also get it via --files equivalent
    .set("spark.files", log4j_config_path)
)

start = time.time()

#Create Spark session in local mode using all cores
spark = SparkSession.builder \
    .config(conf=conf)\
    .getOrCreate()

print("Spark master:", spark.sparkContext.master)
print("Total cores Spark sees:", spark.sparkContext.defaultParallelism)

#Create a large synthetic dataset (e.g., 100 million rows)
num_rows = 99_999
num_partitions = spark.sparkContext.defaultParallelism  # same as CPU cores

df = spark.range(0, num_rows, numPartitions=num_partitions) \
           .withColumn("random_val", rand())

# I want to know the amount of Ram taken by the dataFrame in Mbs
# converting df to rdd rows
rdd = df.rdd.map(lambda row: row.asDict())
# Estimate memory size of one partition
def estimate_partition_size(partition):
    import sys
    size = 0
    for record in partition:
        size += sys.getsizeof(record)
    yield size

partition_sizes = rdd.mapPartitions(estimate_partition_size).collect()
total_bytes = sum(partition_sizes)
total_mb = total_bytes / (1024 * 1024)

#Apply heavy transformations — wide operations
#    Force Spark to use multiple stages and shuffles
aggregated_df = (
    df.withColumn("squared", col("random_val") * col("random_val"))
      .groupBy((col("id") % 100).alias("group"))  # 100 groups
      .avg("squared")                            # aggregation
      .orderBy("group")                          # shuffle operation
)

#Trigger computation (action)
aggregated_df.show()

end = time.time()

time_taken_in_sec = end - start
time_taken_in_min = (end - start) / 60
print(f"""
        Spark task complete!
        Time taken in seconds = {time_taken_in_sec}
        Time taken in minutes = {time_taken_in_min}
        Estimated DataFrame size in memory: {total_mb:.2f} MB
""")


Setting default log level to "DEBUG".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/10/29 12:18:16 INFO Server: jetty-11.0.24; built: 2024-08-26T18:11:22.448Z; git: 5dfc59a691b748796f922208956bd1f2794bcd16; jvm 17.0.15+6-Ubuntu-0ubuntu120.04
25/10/29 12:18:16 INFO Server: Started Server@73b4d8b7{STARTING}[11.0.24,sto=30000] @2048ms
25/10/29 12:18:16 INFO AbstractConnector: Started ServerConnector@3f1256c8{HTTP/1.1, (http/1.1)}{0.0.0.0:4040}
25/10/29 12:18:16 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@10d377b4{/,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Stopped o.s.j.s.ServletContextHandler@10d377b4{/,null,STOPPED,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@16a4e6e3{/jobs,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@2f0d9e56{/jobs/json,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@4bc7147{/jobs/job,null,AVAILABLE,@Spark}
25/10/29 12:18:17 INFO ContextHandler: Started o.s.j.s.ServletC

+-----+-------------------+
|group|       avg(squared)|
+-----+-------------------+
|    0| 0.3279938617519789|
|    1|0.34576483889045606|
|    2|0.32681397541072815|
|    3| 0.3446750728815876|
|    4| 0.3343884808867916|
|    5|0.32564363285727255|
|    6|0.34520534230646555|
|    7|0.34078275580841333|
|    8|0.32975440803719874|
|    9|0.33009214531352643|
|   10| 0.3391300519163157|
|   11| 0.3297460270684851|
|   12|0.34306169008989895|
|   13|0.34458272746086255|
|   14|0.32444943685993444|
|   15|0.33191574922919065|
|   16|0.31318415768603897|
|   17|0.35047172118867703|
|   18|0.33270819405072327|
|   19| 0.3238464271903375|
+-----+-------------------+
only showing top 20 rows

        Spark task complete!
        Time taken in seconds = 7.792145490646362
        Time taken in minutes = 0.1298690915107727
        Estimated DataFrame size in memory: 17.55 MB



### Explaination
- ```log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")```
    - This constructs the absolute path to my Log4j configuration file that lives inside my project directory.
    - if the project tree looks like this 
    ```bash
        pyspark/Spark_programming_model/
        ├── log4j_properties/
        │   └── log4j.properties
        └── your_spark_script.py

    ```
    - Then the log4j_config_path will end_up as something like this ```/home/aditya/pyspark/Spark_programming_model/log4j_properties/log4j.properties```
    - I need the full path because the JVM (which Spark runs on) must know where the log4j file is located.
- ```python
        conf = (
            SparkConf()
            .setAppName("CPU_Stress_Test")
            .setMaster("local[*]")
            # JVM property for log4j v1
            .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            # Ensure executors also get it via --files equivalent
            .set("spark.files", log4j_config_path)
        )
    ```
    - ```SparkConf()```
        - SparkConf creates a Spark configuration object that stores key-value pairs Spark uses when starting up.
        - I will have to pass this to the SparkSession so that Spark knows how to start my local cluster
    - ```.set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")```
        - This is the cruicial part for logging. I am passing a JVM system property (-D...) into Spark's driver process
        - ```spark.driver.extraJavaOptions``` : extra Java options for the Spark Driver JVM
        - ```-Dlog4j.configuration=file:/path/to/log4j.properties``` : tells Log4j where my custom config file is stored
        - Essentially:
            - “Hey Spark driver, when you start your JVM, use this custom Log4j properties file instead of the default one.”
        - **NOTE:** 
            - If you want to set the Log file directory from your Spark application using python then you will have to make changes 
            ```python
                .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            ```
            to
            ```python
                .set("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
            ```
            you will have to set the path to the log_dir like this dynamically
            ```python
                log_dir = os.path.join(project_dir, "log4j_properties", "logs")
                os.makedirs(log_dir, exist_ok=True)  # ensure it exists
            ```
- ```.set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")```
    - Same idea — but applies to executor JVMs (the parallel worker processes that actually execute your tasks).
    - This ensures:
        - Driver logs → follow your log4j config
        - Executor logs → also follow your log4j config
    - Without this line, only your driver would use your custom Log4j configuration, and executor logs might still use the default Spark log setup.
    - **NOTE:** 
        - If you want to set the Log file directory from your Spark application using python then you will have to make changes 
            ```python
                .set("spark.executor.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path}")
            ```
            to
            ```python
                .set("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
            ```
            you will have to set the path to the log_dir like this dynamically
            ```python
                log_dir = os.path.join(project_dir, "log4j_properties", "logs")
                os.makedirs(log_dir, exist_ok=True)  # ensure it exists
            ```
        - In log4j.properties file you will have to change these things also 
            ```python
                log4j.appender.warnErrorFile.File=${custom.log.dir}/warn-error.log

                log4j.appender.debugFile.File=${custom.log.dir}/debug.log
            ```
- ```.set("spark.files", log4j_config_path)```
    - This ensures that **the Log4j.properties** file is distributed to all executors (workers) when the job starts.
    - This ensures that the log4j.properties file is distributed to all executors (workers) when the job starts.
    - In cluster mode, executors run on other nodes — they can’t see your local filesystem.
        - Setting spark.files tells Spark:
            - “Ship this file to every executor node.”
            - Then inside executors, Spark automatically adds the file to their working directory.
### How to configure JVM variables?
- Spark has a complex mechanism to read configuration settings. - Every spark application will look for a SPARK_HOME environment variable ```SPARK_HOME/conf/spark-defaults.conf```.
- If you have SPARK_HOME configured then spark will look for the conf directory of your SPARK_HOME.
- Spark is a JVM based application. It is written in scala and ir runs in a JAVA virtual machine.

### Why Log4j instead of the standard python logger.
We use Log4j instead of the standard python logger because collecting python log files is not integrated with Spark. Spark is designed to work with Log4j and most of the cluster manager also won't give you any issue when managing the Log4j files since it is well supported.
<br>
You can still use the python logger to send log messages to the  console. However if you want to collect your python logs to a central location then will have to configure the remote log handlers and use them in your pyspark program.
<br>
Setting up and using python remote log handlers could be an unnecessary complexity.




## First Spark program

In [20]:
from pyspark.sql import SparkSession

# Log4j related imports 
from pyspark import SparkConf
import os

# Determine if the code is running in a notebook or as a script
try:
    project_dir = os.path.dirname(os.path.abspath(__file__))
except:
    project_dir = os.getcwd()

# Dynamically define the log file directory 
log_dir = os.path.join(project_dir, "log4j_properties", "logs")
# This will create the log directories as per requirement if the directory does not exists
os.makedirs(log_dir,exist_ok=True)

# Set the path to the log4j.properties file where the configurations to the log4j logger is kept for Spark to use
log4j_config_path = os.path.join(project_dir, "log4j_properties","log4j.properties")

# Setting up the Spark configurations for Log4j
config = (
    SparkConf()
    .setAppName("TestApp")
    .setMaster("local[*]")
    # Supplying the directories where the logs must be generated to Spark from python instead of supplying it from log4j.properties file
    .set("spark.driver.extraJavaOptions", f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    .set("spark.executor.extraJavaOptions",f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
    # Making sure all the executors also get the log4j.properties file
    .set("spark.files",log4j_config_path)
)

spark = SparkSession.builder.config(conf=config).getOrCreate()

dataset_file_path = os.path.join(project_dir, "dataset")

spark_df = spark.read.format("csv").option("headers","true").option("inferschema","true").load(f"{dataset_file_path}/sf-fire-calls.csv")

spark_df.show()

+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+-------+-------------+---------+--------------+--------------------+--------------------+------------------+--------------------+--------------------+-------------+---------+
|       _c0|   _c1|           _c2|             _c3|       _c4|       _c5|                 _c6|                 _c7|                 _c8| _c9|   _c10|     _c11|       _c12|_c13|            _c14|    _c15|         _c16|   _c17|         _c18|     _c19|          _c20|                _c21|                _c22|              _c23|                _c24|                _c25|         _c26|     _c27|
+----------+------+--------------+----------------+----------+----------+--------------------+--------------------+--------------------+----+-------+---------+-----------+----+----------------+--------+-------------+--

### Explaination : 
- ```SparkSession`` here is essentially your Spark driver`
    - SparkSession is a singleton object so each spark application can have one and only one active SparkSession.
    - And if you look closely this do make sense because the SparkSession is your driver and you cannot have more than one driver in a spark application.


In [23]:
spark.stop()

## How to get spark session configuration value from a configuration file instead of hard coding it in the application? and How to append DEBUG, WARN-ERROR and INFO log messages in separate files using Log4j.properties file using log filtering?

### First create a ```logger.py``` file

```python
    class Log4j:
    def __init__(self, spark):
        # Get a log4j instance
        log4j = spark._jvm.org.apache.log4j
        # Create a logger attribute
        # put your organization name as a root class 
        root_class = "credencys.aditya.spark"
        conf = spark.sparkContext.getConf()
        app_name = conf.get("spark.app.name")
        self.logger = log4j.LogManager.getLogger(root_class + "." + app_name)

    def warn(self,message):
        self.logger.warn(message)
    
    def info(self,message):
        self.logger.info(message)
    
    def error(self, message):
        self.logger.error(message)
    
    def debug(self,message):
        self.logger.debug(message)
     
```

#### Explaination :
- `spark` is your active **SparkSession** (created by `SparkSession.builder.getOrCreate()`).
- The attribute `_jvm` gives you access to the **Java Virtual Machine (JVM) gateway** — the place where PySpark communicates with the underlying Java Spark runtime. So `spark._jvm` is your **Python handle to the JVM world**
- From there, you can access any Java class loaded by Spark, such as:
- This literally is referencing the Java package:
```python
    spark._jvm.org.apache.log4j
```
- You can think of it like importing ```org.apache.log4j``` from Java into Python, using the Py4J bridge.
- This line gives you access to all Log4j classes (```LogManager```, ```Logger```, etc) from Python
- **root_class = "credencys.aditya.spark"**
    - This is just a **namespace prefix** — a naming convention for your logs.
    - Loggers in Log4j are hierarchical, based on **dot-separated names**
    - For example:
        - `credencys` (root)
        - `credencys.aditya`
        - `credencys.aditya.spark.SparkLoggerDemo`
    - Log4j can apply different logging rules (appenders, levels) to different parts of this hierarchy.
    - **Purpose:**  
    - You’re defining a **root logger name prefix** for your custom application or organization — e.g., all your Spark apps will have logger names starting with `"credencys.aditya.spark"`.
- **conf = spark.sparkContext.getConf()**
    - spark.sparkContext is the core SparkContext — the heart of every Spark application.
    - .getConf() returns the SparkConf object — the configuration object that holds all Spark settings (like master URL, app name, executor memory, etc.).
    - Retrieve the configuration so you can read app-specific properties like the app name.
- **app_name = conf.get("spark.app.name")**
    - This fetches the value of the configuration property `spark.app.name` from the SparkConf.
    - That’s the name you usually set when creating the Spark session, e.g.:
        - ```python
            spark = SparkSession.builder.appName("SparkLoggerDemo").getOrCreate()
          ```
    - Purpose : 
        - Get the Spark application’s name so you can include it in the logger name — this helps you distinguish logs from multiple Spark jobs.
- **self.logger = log4j.LogManager.getLogger(root_class + "." + app_name)**
    - You are now calling into the **Java Log4j LogManager** via the Py4J bridge.
    - `LogManager.getLogger(name)` returns a **Logger** instance identified by that name.
    - So if `root_class` = `"credencys.aditya.spark"` and `app_name` = `"SparkLoggerDemo"`, your full logger name will be `"credencys.aditya.spark.SparkLoggerDemo"`
    - **Purpose:**
        - This create a **Java Log4j logger instance**, which follows the configuration rules you've defined in your `log4j.properties` (appenders, thresholds, etc.).
        - It’s stored in `self.logger` so you can reuse it for `info()`, `warn()`, etc.

### Second create a ```util.py``` file
This is a spark application configuration file with a name ```spark.conf```. This file holds the value related to a spark application configuration.
This lets you define Spark settings outside your Python code — which is a best practice for maintainability.
```bash
[SPARK_APP_CONFIGS]
saprk.app.name = SparkLoggerDemo
spark.master = local[3]
```
This ```get_spark_app_config``` function reads your custom configuration file spark.conf, extract the Spark settings from it, and return a properly configured SparkConf object that can be passed into SparkSession.builder.config(conf=...)
```python
    import configparser
    from pyspark import SparkConf
    import os


    """
    This function will load the configuration from spark.conf file and return a spark conf object
    """
    def get_spark_app_config():
        spark_conf = SparkConf()
        config = configparser.ConfigParser()
        # Read the spark.conf file from the dirtectory
        config.read(os.path.join(os.getcwd(),"..","spark.conf"))

        # Loop through the configs and set it to the spark conf
        for (key, val) in config.items("SPARK_APP_CONFIGS"):
            spark_conf.set(key,val)
        return spark_conf
```
#### Explaination :
- `SparkConf` is a **PySpark class** that holds configuration settings for a Spark application.
    - It stores key–value pairs like:
    - `spark.app.name` — the name of your Spark job
    - `spark.master` — the cluster manager or mode (`local[*]`, `yarn`, etc.)
    - `spark.executor.memory`, etc.
    - When you create a SparkSession, you can pass this `SparkConf` to configure how Spark will run.
    - Purpose : Create an empty SparkConf object that you can populate with key–value pairs from your configuration file.
- ```config = configparser.ConfigParser()```
    - `configparser` is a **Python standard library module** used for reading `.ini`-style configuration files.
    - These files have sections like `[SECTION_NAME]`, followed by key–value pairs.
    Example:
    ```ini
        [SPARK_APP_CONFIGS]
        spark.app.name = SparkLoggerDemo
        spark.master = local[3]
    ```
    - When you call ConfigParser(), you create an object that can:
        - Read .ini files
        - Access their contents as dictionaries
        - Handle comments, sections, etc.
    - Purpose : Prepare to parse your custom configuration file (spark.conf).
- ```config.read(os.path.join(os.getcwd(), "..", "spark.conf"))``` 
    - `os.getcwd()` → returns the **current working directory** where your Python process is running.
    - `os.path.join(os.getcwd(), "..", "spark.conf")` → constructs a **path** to the `spark.conf` file **one directory above** the current working directory (`..` means parent directory).
    - `config.read(<path>)` → tells `ConfigParser` to read and parse the file at that path.
    - Purpose : Load the contents of `spark.conf` from the parent directory into the `config` object.
- ```python
    for (key, val) in config.items("SPARK_APP_CONFIGS"):
        spark_conf.set(key, val)
  ```
    - Now you’re looping through all the key–value pairs inside the section [SPARK_APP_CONFIGS] in your file.
    - config.items("SPARK_APP_CONFIGS") returns a list like: 
    ```python
        [("spark.app.name", "SparkLoggerDemo"), ("spark.master", "local[3]")]

    ```
    - For each pair:
        - key = "spark.app.name", val = "SparkLoggerDemo"
        - spark_conf.set(key, val) → sets that configuration in your SparkConf object.
    - Purpose: Transfer all Spark app configurations from the file into a SparkConf object.
- Finally, the function returns the fully populated SparkConf object, which can be directly used when initializing Spark:
```python
    conf = get_saprk_app_config()
    spark = (
        SparkSession.builder
        .config(conf=conf)
        .getOrCreate()
    )
```
Purpose: Return the ready-to-use configuration for your SparkSession.

### Third create a ```log4j.properties``` file
```bash
# Root logger configuration
log4j.rootCategory=INFO, console

# Console output
log4j.appender.console=org.apache.log4j.ConsoleAppender
log4j.appender.console.layout=org.apache.log4j.PatternLayout
log4j.appender.console.layout.ConversionPattern=%d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n

# ======================================================
# INFO -> logs/info.log
# ======================================================
log4j.appender.infoFile=org.apache.log4j.FileAppender
log4j.appender.infoFile.File=${custom.log.dir}/info.log
log4j.appender.infoFile.Append=true
log4j.appender.infoFile.Threshold=INFO
log4j.appender.infoFile.layout=org.apache.log4j.PatternLayout
log4j.appender.infoFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
# filter section: This will exclude warn and error logs only info logs will be appended
log4j.appender.infoFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
log4j.appender.infoFile.filter.a.LevelToMatch=INFO
log4j.appender.infoFile.filter.a.AcceptOnMatch=true
log4j.appender.infoFile.filter.b=org.apache.log4j.varia.DenyAllFilter

# WARN + ERROR -> logs/warn-error.log
# This will allow me to use the logger.py file
log4j.appender.warnErrorFile=org.apache.log4j.FileAppender
log4j.appender.warnErrorFile.File=${custom.log.dir}/warn-error.log
log4j.appender.warnErrorFile.Append=true
log4j.appender.warnErrorFile.Threshold=WARN
log4j.appender.warnErrorFile.layout=org.apache.log4j.PatternLayout
log4j.appender.warnErrorFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n

# DEBUG -> logs/debug.log
log4j.appender.debugFile=org.apache.log4j.FileAppender
log4j.appender.debugFile.File=${custom.log.dir}/debug.log
log4j.appender.debugFile.Append=true
log4j.appender.debugFile.Threshold=DEBUG
log4j.appender.debugFile.layout=org.apache.log4j.PatternLayout
log4j.appender.debugFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
# Filter: only DEBUG
log4j.appender.debugFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
log4j.appender.debugFile.filter.a.LevelToMatch=DEBUG
log4j.appender.debugFile.filter.a.AcceptOnMatch=true
log4j.appender.debugFile.filter.b=org.apache.log4j.varia.DenyAllFilter

# Loggers
# log4j.logger.org.apache.spark=DEBUG, debugFile, warnErrorFile
# log4j.additivity.org.apache.spark=false

# log4j.logger.org.apache=INFO, warnErrorFile
# log4j.additivity.org.apache=false

# Here I am defining a common root logger definitions for appenders
log4j.logger.credencys.aditya.spark=DEBUG, debugFile, warnErrorFile, infoFile
log4j.additivity.credencys.aditya.spark=false
```
#### Explaination :
- ```log4j.rootCategory=INFO, console```
    - log4j.rootCategory (aka root logger): sets the default logging level and the default appenders for all loggers that don't have an explicit configuration.
    - Here the root level is INFO (so by default events with level INFO and above are considered), and console is the appender attached to the root logger.
- ```bash
    # Console output
    log4j.appender.console=org.apache.log4j.ConsoleAppender
    log4j.appender.console.layout=org.apache.log4j.PatternLayout
    log4j.appender.console.layout.ConversionPattern=%d{yy/MM/dd HH:mm:ss} %p %c{1}: %m%n
    ```
    - log4j.appender.console: declares an appender named console which is an instance of ConsoleAppender (writes to STDOUT).
    - layout=PatternLayout: controls the format of each log line.
    - ConversionPattern: the actual format string:
        - %d{yy/MM/dd HH:mm:ss} — timestamp (year/month/day hour:minute:second)
        - %p — log level (INFO/WARN/DEBUG/ERROR)
        - %c{1} — logger name, shortened to the last component
        - : %m — the log message
        - %n — newline
- ```bash
    # ======================================================
    # INFO -> logs/info.log
    # ======================================================
    log4j.appender.infoFile=org.apache.log4j.FileAppender
    log4j.appender.infoFile.File=${custom.log.dir}/info.log
    log4j.appender.infoFile.Append=true
    log4j.appender.infoFile.Threshold=INFO
    log4j.appender.infoFile.layout=org.apache.log4j.PatternLayout
    log4j.appender.infoFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
    # filter section: This will exclude warn and error logs only info logs will be appended
    log4j.appender.infoFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
    log4j.appender.infoFile.filter.a.LevelToMatch=INFO
    log4j.appender.infoFile.filter.a.AcceptOnMatch=true
    log4j.appender.infoFile.filter.b=org.apache.log4j.varia.DenyAllFilter
    ```
    - log4j.appender.infoFile: defines a FileAppender named infoFile.
    - File=${custom.log.dir}/info.log: output path uses the system property custom.log.dir. This must be set when the JVM starts (e.g. -Dcustom.log.dir=/path/to/logs), otherwise file path may be wrong.
    - Append=true: append to existing file instead of overwriting.
    - Threshold=INFO: basic threshold — the appender will only consider events with level ≥ INFO. (Note: with filters below, this is a coarse filter; the LevelMatchFilter further restricts.)
    - Filters:
        - LevelMatchFilter with LevelToMatch=INFO and AcceptOnMatch=true → accept only messages whose level is exactly INFO.
        - DenyAllFilter following it → deny everything else that reaches this appender.
    - Combined effect: although Threshold=INFO would accept INFO+WARN+ERROR by default, the LevelMatchFilter forces the appender to only write exact INFO-level events to info.log.
- ```bash
    # WARN + ERROR -> logs/warn-error.log
    # This will allow me to use the logger.py file
    log4j.appender.warnErrorFile=org.apache.log4j.FileAppender
    log4j.appender.warnErrorFile.File=${custom.log.dir}/warn-error.log
    log4j.appender.warnErrorFile.Append=true
    log4j.appender.warnErrorFile.Threshold=WARN
    log4j.appender.warnErrorFile.layout=org.apache.log4j.PatternLayout
    log4j.appender.warnErrorFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
    ```
    - warnErrorFile: another FileAppender that writes to warn-error.log.
    - Threshold=WARN: this appender will accept events with level WARN and above (i.e., WARN, ERROR, FATAL).
    - There are no filters here, so WARN and ERROR events will be written to this file.
- ```bash
    # DEBUG -> logs/debug.log
    log4j.appender.debugFile=org.apache.log4j.FileAppender
    log4j.appender.debugFile.File=${custom.log.dir}/debug.log
    log4j.appender.debugFile.Append=true
    log4j.appender.debugFile.Threshold=DEBUG
    log4j.appender.debugFile.layout=org.apache.log4j.PatternLayout
    log4j.appender.debugFile.layout.ConversionPattern=%d{yyyy-MM-dd HH:mm:ss} %-5p %c{1}:%L - %m%n
    # Filter: only DEBUG
    log4j.appender.debugFile.filter.a=org.apache.log4j.varia.LevelMatchFilter
    log4j.appender.debugFile.filter.a.LevelToMatch=DEBUG
    log4j.appender.debugFile.filter.a.AcceptOnMatch=true
    log4j.appender.debugFile.filter.b=org.apache.log4j.varia.DenyAllFilter
    ```
    - debugFile: FileAppender for debug output.
    - Threshold=DEBUG: would normally accept DEBUG and all higher levels (DEBUG, INFO, WARN, ERROR).
    - LevelMatchFilter with LevelToMatch=DEBUG + DenyAllFilter ensures that this appender accepts only exact DEBUG-level events, rejecting INFO/WARN/ERROR.
- ```python
        # Here I am defining a common root logger definitions for appenders
        log4j.logger.credencys.aditya.spark=DEBUG, debugFile, warnErrorFile, infoFile
        log4j.additivity.credencys.aditya.spark=false
    ```
    - ```log4j.logger.credencys.aditya.spark=DEBUG, debugFile, warnErrorFile, infoFile```
        - Defines a logger with name credencys.aditya.spark.
        - The logger level is DEBUG (so this logger will process events at DEBUG and above).
        - It is explicitly attached to the appenders: debugFile, warnErrorFile, and infoFile.
            - This means when this logger emits a message, it is handed to each attached appender; each appender then decides (via Threshold and filters) whether to write the event to its file.
    - ```log4j.additivity.credencys.aditya.spark=false```
        - Disables additivity for that logger: prevents log events from also being passed up to parent loggers (for example the root logger) and avoid duplicate logging to root appenders (like console) unless desired.
#### How the pieces interact (summary of runtime behavior)
- When your code does logger.debug("x"), the logger credencys.aditya.spark sees a DEBUG event and passes it to its three appenders.
    - debugFile → LevelMatchFilter accepts DEBUG → writes to debug.log.
    - infoFile → LevelMatchFilter denies (not INFO) → nothing written.
    - warnErrorFile → Threshold=WARN (DEBUG < WARN) → ignored.

- When you log logger.info("x"):
    - infoFile → filter accepts INFO → write to info.log.
    - debugFile → filter denies (not DEBUG).
    - warnErrorFile → threshold WARN (INFO < WARN) → ignored.

- When you log logger.warn("x"):
    - warnErrorFile → threshold allows WARN → write to warn-error.log.
    - infoFile → LevelMatchFilter denies (not INFO).
    - debugFile → LevelMatchFilter denies (not DEBUG).

### Fourth create a ```main_app.py``` file
```python
from pyspark.sql import SparkSession
# import related to logging
from lib.logger import Log4j
# import related to custom spark configurations
from lib.utils import get_spark_app_config
# logging related imports 
import os

if __name__ == "__main__":
    # logging related logic
    # Get the current project's directory
    project_dir = os.path.dirname(os.path.abspath(__file__))
    # Get the Log4j.properties file directory
    log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")
    # Save the directory where the generated log files must reside
    log_dir = os.path.join(project_dir, "log4j_properties", "logs")
    # Create the directory where the log files must be kept if not present
    os.makedirs(log_dir, exist_ok=True)

    conf = get_spark_app_config()
    spark = (
        SparkSession
        .builder
        .config(conf=conf)
        .config("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .getOrCreate()
    )

    logger = Log4j(spark)
    logger.warn(">>>Starting SparkLoggerDemo")
    conf_out = spark.sparkContext.getConf()
    logger.info(">>>{conf_out}")
    logger.debug(">>Testing the debug messages")
    logger.warn(">>>Ending SparkLoggerDemo")
    logger.error(">>>System.out.println")
    spark.stop()
```
- ```from pyspark.sql import SparkSession```
    - Imports the SparkSession builder API from PySpark. SparkSession is the entry point to use Spark SQL and to create the Spark application.
    - SparkSession is the driver for a spark application we can say that because each spark application can have one and only one active SparkSession.
- ```import os```
    - Imports the standard library os module for filesystem and path operations.
- ```if __name__ == "__main__":```
    - Standard Python idiom. Ensures the following block runs only when the script is executed as the main program (not when imported as a module).
- ```project_dir = os.path.dirname(os.path.abspath(__file__))```
    - __file__ is the path of the current script.
    - os.path.abspath(__file__) resolves it to an absolute path
    - os.path.dirname(...) returns the directory containing the script.
    - Result: project_dir holds the absolute directory path of your project/script — used for building paths relative to the project.
- ```log4j_config_path = os.path.join(project_dir, "log4j_properties", "log4j.properties")```
    - Constructs the full path to the log4j.properties file that configures Log4j. Example:
    - ```/path/to/project/log4j_properties/log4j.properties```
- ```log_dir = os.path.join(project_dir, "log4j_properties", "logs")```
    - Constructs the directory path where log files will be written (e.g. .../log4j_properties/logs).
- ```os.makedirs(log_dir, exist_ok=True)```
    - Creates log_dir if it doesn’t exist. exist_ok=True prevents an error if the directory already exists.
- ```conf = get_spark_app_config()```
    - Calls your helper to read spark.conf and return a SparkConf object with entries like spark.app.name and spark.master. This conf will be used to configure the SparkSession.
- ```python
    spark = (
        SparkSession
        .builder
        .config(conf=conf)
        .config("spark.driver.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .config("spark.executor.extraJavaOptions",
                f"-Dlog4j.configuration=file:{log4j_config_path} -Dcustom.log.dir={log_dir}")
        .getOrCreate()
    )

    ```
    - .builder → entry builder for creating a SparkSession.
    - .config(conf=conf) → supplies the SparkConf you built from spark.conf. This applies the settings inside conf.
    - .config("spark.driver.extraJavaOptions", ...) → passes JVM options to the driver process. Here you set two -D system properties:
        - -Dlog4j.configuration=file:{log4j_config_path} tells the JVM where to find the log4j.properties file (explicit file path).
        - -Dcustom.log.dir={log_dir} defines a JVM system property custom.log.dir that your log4j.properties uses as ${custom.log.dir} in file paths.
        - Using this ensures Log4j can resolve where to write info.log, debug.log, etc.
    - .config("spark.executor.extraJavaOptions", ...) → does the same but for executor JVMs (useful in distributed/cluster mode so executors use same log4j config). In local mode executors may still be separate JVM processes if you use multiple local threads.
    - .getOrCreate() → actually creates or retrieves the SparkSession.
- NOTE:
    - The spark.driver.extraJavaOptions and spark.executor.extraJavaOptions must be set before SparkSession is created so the JVM starts with those system properties.
    - If you run on a cluster, spark.executor.extraJavaOptions ensures executors pick up the same logging configuration. In client-mode standalone testing, driver logging configuration is most important.
- ```logger = Log4j(spark)```
    - Instantiates your Log4j wrapper with the spark session so the wrapper can access spark._jvm and create a Java Log4j logger (e.g. credencys.aditya.spark.<app_name>).
- ```logger.warn(">>>Starting SparkLoggerDemo")```
    - Calls the wrapper’s .warn() which forwards the string to the underlying JVM Log4j logger at WARN level. The message will be routed to appenders based on log4j.properties.
- ```conf_out = spark.sparkContext.getConf()```
    - Reads the active SparkConf from the running SparkContext (this will include config keys/values). conf_out is a SparkConf object (or representation) that you might want to print/log for debugging.
- ```logger.debug(">>Testing the debug messages")```
    - Logs a DEBUG-level message via Log4j. This will be handled by your configured appenders according to filters and thresholds (e.g., debug.log if you configured LevelMatchFilter for DEBUG).
- ```logger.warn(">>>Ending SparkLoggerDemo")```
    - Another WARN-level message (writes to warn-error.log per your configuration).
- ```logger.error(">>>System.out.println")```
    - Logs at ERROR level. It will be routed to your warn-error appender (or any appender configured for ERROR) according to log4j.properties.
- ```spark.stop()```
    - Gracefully stops the SparkSession (and underlying SparkContext), releasing resources. Always call stop() at the end of your application to shut down the Spark JVM and free cluster resources.
    